In [ ]:
# =========================================
# sustainableStartUps -PROJECT DASHBOARD GENERATOR (version1.0)
# =========================================

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from google.colab import files
import requests
from transformers import pipeline
from time import sleep

# ===============================
# STEP 1: LOAD DATASET WITH UPLOAD PROMPT
# ===============================
file_path = "/content/startup-sustainability-dashboard/datasets/startup_dataset_curated.csv"
if not os.path.exists(file_path):
    print("Dataset not found. Please upload startup_dataset_curated.csv")
    uploaded = files.upload()
    for filename in uploaded.keys():
        file_path = filename

df = pd.read_csv(file_path)
print(f"Loaded dataset: {file_path}")

# ===============================
# STEP 1A: ASK FOR NEWSAPI KEY
# ===============================
NEWS_API_KEY = input("Enter your NewsAPI key: ").strip()
NEWS_API_URL = "https://newsapi.org/v2/everything"

# ===============================
# STEP 2: FILTER FOR COMPLETE DATA
# ===============================
df_complete = df[df['Data Completeness'] == 'Complete']
incomplete_count = len(df) - len(df_complete)

# Split revenue into columns
rev_split = df_complete["Revenue FY21–23 (INR Cr)"].str.split("/", expand=True)
df_complete["Revenue_FY21"] = pd.to_numeric(rev_split[0], errors="coerce")
df_complete["Revenue_FY22"] = pd.to_numeric(rev_split[1], errors="coerce")
df_complete["Revenue_FY23"] = pd.to_numeric(rev_split[2], errors="coerce")

# ===============================
# STEP 2B: AUTOMATED SENTIMENT ANALYSIS WITH ERROR HANDLING
# ===============================
!pip install transformers --quiet
sentiment_model = pipeline("sentiment-analysis")

def fetch_latest_headline(startup):
    """Fetches the latest news headline for a given startup using NewsAPI."""
    params = {
        "q": startup,
        "apiKey": NEWS_API_KEY,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": 1
    }
    try:
        response = requests.get(NEWS_API_URL, params=params)
        if response.status_code != 200:
            return None
        data = response.json()
        if "articles" in data and data["articles"]:
            return data["articles"][0]["title"]
        else:
            return None
    except Exception:
        return None

# Fetch headlines & run sentiment
headlines = []
sentiments = []
newsapi_failed = False
for startup in df_complete['Startup']:
    headline = fetch_latest_headline(startup)
    if headline:
        sentiment = sentiment_model(headline)[0]['label']
    else:
        sentiment = "Not Available"
    headlines.append(headline if headline else "No recent news found")
    sentiments.append(sentiment)
    sleep(1)  # avoid hitting API rate limits

# Detect API issues
if all(s == "Not Available" for s in sentiments):
    newsapi_failed = True

df_complete['Latest Headline'] = headlines
df_complete['Media Sentiment'] = sentiments

if newsapi_failed:
    sentiment_note = "<p style='color:red;'><i>Sentiment analysis could not be performed due to API limitations or invalid key.</i></p>"
else:
    sentiment_note = f"<p><i>Sentiment analysis performed for <b style='color:darkblue;'>{sum(df_complete['Media Sentiment']!='Not Available')}</b> out of <b style='color:darkblue;'>{len(df_complete)}</b> startups. Startups with no recent news are marked as 'Not Available'.</i></p>"

# Export updated dataset
# df_complete.to_csv("cleaned_dataset_with_sentiment.csv", index=False)
df_complete.to_csv("/content/startup-sustainability-dashboard/datasets/cleaned_dataset_with_sentiment.csv", index=False)
sector_summary.to_csv("/content/startup-sustainability-dashboard/datasets/sector_summary.csv", index=False)
with open("/content/startup-sustainability-dashboard/outputs/interactive_dashboard_curated.html", "w") as f:
    f.write(final_html)

# ===============================
# STEP 3: GENERATE KEY INSIGHTS
# ===============================
insights = []
top_valuation_sector = df_complete.groupby("Sector")["Valuation (USD B)"].mean().idxmax()
insights.append(f"<b style='color:darkblue;'>{top_valuation_sector}</b> startups have the highest average valuations, reflecting strong investor confidence.")
growth = (df_complete.groupby("Sector")["Revenue_FY23"].mean() - df_complete.groupby("Sector")["Revenue_FY21"].mean()).idxmax()
insights.append(f"<b style='color:darkblue;'>{growth}</b> sector shows the highest revenue growth from FY21 to FY23.")
profit_counts = df_complete[df_complete['Profitability'] == 'Profitable']['Sector'].value_counts()
if not profit_counts.empty:
    top_profitable = profit_counts.idxmax()
    insights.append(f"<b style='color:darkblue;'>{top_profitable}</b> has the most profitable startups in this dataset.")
esg_sector = df_complete[df_complete['ESG Disclosed'] == 'Yes']['Sector'].value_counts().idxmax()
insights.append(f"ESG adoption is strongest among <b style='color:darkblue;'>{esg_sector}</b> startups, aligning with regulatory and customer expectations.")
ipo_count = df_complete[df_complete['IPO-Listed'] == 'Yes'].shape[0]
insights.append(f"There are <b style='color:darkblue;'>{ipo_count}</b> IPO‑listed startups in this curated dataset, indicating market maturity.")
insights_html = "<ul>" + "".join([f"<li>{i}</li>" for i in insights]) + "</ul>"

# ===============================
# STEP 4: SECTOR-WISE SUMMARY
# ===============================
sector_summary = df_complete.groupby("Sector").agg({
    "Valuation (USD B)": "mean",
    "Funding (USD B)": "mean",
    "Revenue_FY23": "mean"
}).reset_index()
sector_summary.rename(columns={
    "Valuation (USD B)": "Avg Valuation (USD B)",
    "Funding (USD B)": "Avg Funding (USD B)",
    "Revenue_FY23": "Avg Revenue FY23 (INR Cr)"
}, inplace=True)
sector_summary.to_csv("sector_summary.csv", index=False)

# ===============================
# STEP 5: CREATE DASHBOARD
# ===============================
fig_dashboard = make_subplots(
    rows=3, cols=2,
    subplot_titles=("Funding vs Revenue", "Business Model Distribution",
                    "Profitability by Sector", "ESG Disclosure by Sector",
                    "Profitability vs ESG Heatmap", "Revenue Growth by Size"),
    specs=[[{}, {"type": "domain"}], [{}, {}], [{}, {}]],
    vertical_spacing=0.15,
    horizontal_spacing=0.12
)

# Funding vs Revenue
fig1 = px.scatter(df_complete, x="Funding (USD B)", y="Revenue_FY23", color="Sector",
                  size="Valuation (USD B)", hover_data=["Startup", "Stage", "ESG Disclosed"])
fig1.update_layout(xaxis_title="Funding (USD B)", yaxis_title="Revenue FY23 (INR Cr)")
for trace in fig1.data:
    fig_dashboard.add_trace(trace, row=1, col=1)

# Business model distribution
fig2 = px.pie(df_complete, names="Business Model")
for trace in fig2.data:
    fig_dashboard.add_trace(trace, row=1, col=2)

# Profitability by sector
fig3 = px.histogram(df_complete, x="Sector", color="Profitability", barmode="group")
fig3.update_layout(xaxis_title="Sector", yaxis_title="Number of Startups")
for trace in fig3.data:
    fig_dashboard.add_trace(trace, row=2, col=1)

# ESG disclosure by sector
fig4 = px.histogram(df_complete, x="Sector", color="ESG Disclosed", barmode="group")
fig4.update_layout(xaxis_title="Sector", yaxis_title="Number of Startups")
for trace in fig4.data:
    fig_dashboard.add_trace(trace, row=2, col=2)

# Profitability vs ESG heatmap with isolated colorbar
heat_data = pd.crosstab(df_complete['Profitability'], df_complete['ESG Disclosed'])
fig5 = go.Heatmap(
    z=heat_data.values,
    x=heat_data.columns,
    y=heat_data.index,
    colorscale="YlGnBu",
    showscale=True,
    colorbar=dict(
        title="Startup Count",
        titleside="top",
        orientation="v",
        x=1.15,
        len=0.5
    )
)
fig_dashboard.add_trace(fig5, row=3, col=1)

# Revenue growth by size
growth_data = df_complete.groupby("Size Category")[["Revenue_FY21","Revenue_FY23"]].mean().reset_index()
fig6 = px.bar(growth_data, x="Size Category", y="Revenue_FY23", color="Size Category")
fig6.update_layout(xaxis_title="Size Category", yaxis_title="Avg Revenue FY23 (INR Cr)")
for trace in fig6.data:
    fig_dashboard.add_trace(trace, row=3, col=2)

fig_dashboard.update_layout(
    height=1900,
    width=1250,
    title_text="Sustainable Business Models Dashboard: Curated Insights",
    title_x=0.5,
    title_font=dict(size=22, color='darkblue'),
    margin=dict(l=80, r=250, t=180, b=80),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="center",
        x=0.5
    )
)

# ===============================
# STEP 6: TEXTUAL SECTIONS
# ===============================
intro_text = """
<h2>Executive Summary</h2>
<p style="text-align:justify;">
This analysis examines 20 curated <b style='color:darkblue;'>Indian internet-based startups</b> across EdTech, FinTech, E‑commerce, and SaaS sectors
to evaluate their business model sustainability.
Findings show E‑commerce leads in valuation and growth, while SaaS startups demonstrate higher profitability.
ESG adoption is strongest in FinTech, reflecting regulatory and market alignment.
</p>
<h2>Methodology & Scope</h2>
<p style="text-align:justify;">
The startups were selected based on IPO‑listing or unicorn status with credible, publicly available financial and ESG data from IPO filings,
annual reports, and reputed business databases. The analysis focuses on multi‑year revenues (FY21–FY23), funding, valuations,
profitability, and ESG disclosures as indicators of sustainable business models.
</p>
<h2>Purpose & Topic Alignment</h2>
<p style="text-align:justify;">
This project aligns with the topic <b>‘An analysis of sustainable business models among internet business start‑ups in India’</b>,
assessing sustainability through financial resilience, growth patterns, and ESG adoption.
</p>
"""

# Chart subtitles
chart_subtitles = """
<div style="margin-top:20px;">
<h3>Chart Notes:</h3>
<ul>
<li><b>Funding vs Revenue:</b> X = Funding in USD B, Y = Revenue in INR Cr, bubble size = valuation, colors = sectors.</li>
<li><b>Business Model Distribution:</b> Shows proportion of startups using different business models.</li>
<li><b>Profitability by Sector:</b> Number of startups grouped by profitability within each sector.</li>
<li><b>ESG Disclosure by Sector:</b> Number of startups with ESG disclosures, grouped by sector.</li>
<li><b>Profitability vs ESG Heatmap:</b> Startup count by profitability status vs ESG disclosure.</li>
<li><b>Revenue Growth by Size:</b> Average revenue in FY23 grouped by company size category.</li>
</ul>
</div>
"""

# Sentiment visuals
if not newsapi_failed and (df_complete['Media Sentiment'] != "Not Available").any():
    sentiment_fig = px.pie(df_complete[df_complete['Media Sentiment'] != "Not Available"],
                           names='Media Sentiment', title='Media Sentiment Distribution')
    sentiment_bar = px.bar(df_complete[df_complete['Media Sentiment'] != "Not Available"],
                           x='Sector', color='Media Sentiment', title='Sentiment by Sector', barmode='group')
    sentiment_charts_html = sentiment_fig.to_html(include_plotlyjs=False, full_html=False) + sentiment_bar.to_html(include_plotlyjs=False, full_html=False)
else:
    sentiment_charts_html = ""

# Generate simple interpretation text for sentiment
if not newsapi_failed:
    sentiment_counts = df_complete['Media Sentiment'].value_counts()
    positive_count = sentiment_counts.get('POSITIVE', 0)
    negative_count = sentiment_counts.get('NEGATIVE', 0)
    neutral_count = sentiment_counts.get('NEUTRAL', 0)
    most_positive_sector = df_complete[df_complete['Media Sentiment'] == 'POSITIVE']['Sector'].mode()[0] if positive_count > 0 else "N/A"
    most_negative_sector = df_complete[df_complete['Media Sentiment'] == 'NEGATIVE']['Sector'].mode()[0] if negative_count > 0 else "N/A"

    sentiment_interpretation = f"""
    <p><b>Sentiment Insights:</b> Out of <b style='color:darkblue;'>{len(df_complete)}</b> startups,
    <b style='color:darkblue;'>{positive_count}</b> have a positive media sentiment,
    <b style='color:darkblue;'>{negative_count}</b> have a negative sentiment,
    and <b style='color:darkblue;'>{neutral_count}</b> are neutral.
    <b style='color:darkblue;'>{most_positive_sector}</b> sector startups lead in positive sentiment,
    while <b style='color:darkblue;'>{most_negative_sector}</b> sector startups show the highest negative sentiment.
    This reflects current public/media perceptions of these companies, which may indicate market confidence or emerging challenges.</p>
    """
else:
    sentiment_interpretation = ""

sentiment_section = f"""
<hr>
<h2>Media Sentiment Insights</h2>
<p>This section analyzes media sentiment for the curated startups using Hugging Face NLP and NewsAPI.
It provides an overview of how these startups are perceived in recent news coverage.</p>
{sentiment_note}
{sentiment_interpretation}
{sentiment_charts_html}
"""

# Sources section with note
sources_html = "<h3>Sources & Notes:</h3><p><i>Click to expand for source references per startup.</i></p>"
for _, row in df_complete.iterrows():
    sources_html += f"<details><summary><b>{row['Startup']}</b></summary><p>{row['Source Notes']}</p></details>"

# Data dictionary section
data_dict = """
<hr>
<h3>Data Dictionary:</h3>
<ul>
<li><b>Valuation (USD B):</b> Estimated market value of the startup in billions of USD.</li>
<li><b>Funding (USD B):</b> Total funding raised by the startup in billions of USD.</li>
<li><b>Revenue FY21–FY23 (INR Cr):</b> Reported revenue for FY21 to FY23 in INR crores.</li>
<li><b>Profitability:</b> Indicates if the startup is profitable, loss-making, or near breakeven.</li>
<li><b>ESG Disclosed:</b> Whether the startup reports Environmental, Social, and Governance initiatives.</li>
<li><b>Media Sentiment:</b> Sentiment derived from recent news coverage using NLP.</li>
<li><b>IPO-Listed:</b> Whether the startup is listed on a stock exchange.</li>
</ul>
"""

footer = f"""
<hr>
<p style="text-align:center; font-size:12px; color:gray;">
Prepared by: Vaisakh<br>
Disclaimer: This analysis includes <b style='color:darkblue;'>{len(df_complete)}</b> startups with complete financial and operational data.
<b style='color:darkblue;'>{incomplete_count}</b> startups were excluded due to incomplete or unverifiable data.
This dashboard is prepared for academic purposes only.
</p>
"""

# ===============================
# STEP 7: COMBINE INTO FINAL HTML
# ===============================
dashboard_html = fig_dashboard.to_html(include_plotlyjs='cdn', full_html=False)
final_html = f"""
<html>
<head><title>Sustainable Business Models Dashboard</title></head>
<body style="font-family:Arial; line-height:1.6; margin:40px;">
<h1 style="text-align:center; color:darkblue;">Sustainable Business Models Dashboard</h1>
{intro_text}
<hr>
<h2>Key Insights</h2>
{insights_html}
<hr>
{dashboard_html}
{chart_subtitles}
{sentiment_section}
{data_dict}
<hr>
{sources_html}
{footer}
</body>
</html>
"""

with open("interactive_dashboard_curated.html", "w") as f:
    f.write(final_html)

print("Interactive dashboard saved as interactive_dashboard_curated.html")


optional for Jupyter explorations - notebooks/sustainableStartUps_Analysis.ipynb    
new script in startup-sustainability-dashboard/analysis.py 